# Agent Running Test 
### Prerequisite
1. Install Ollama from https://ollama.com
2. Start Ollama `ollama run qwen3:4b`
3. install docker

In [1]:
import torch

if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected - PyTorch is using CPU-only version or no CUDA drivers")

GPU device: Quadro RTX 8000


In [2]:
import os

os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

import transformers

transformers.logging.set_verbosity_error()

In [3]:
example_path = 'agent_using_our_model.py'

## Python Interpreter

In [4]:
!python {example_path}

Using device: cuda:0
The grounding tool detected two bounding boxes for the target entity "cat" in the image. Here are the coordinates:

1. **First Box**: Top-left (9, 54) to Bottom-right (316, 474)
2. **Second Box**: Top-left (346, 23) to Bottom-right (639, 372)

These boxes represent the detected positions of cats in the image with a confidence threshold of 0.3.


## Run the agent and tool teogether locally

In [5]:
# Removed pip_install parameter from decorators in {example_path}
import ast 
import os
import tempfile


with open(example_path, "r") as f:
    source_code = f.read()
    mod = ast.parse(source_code)

# Remove env keyword argument
for node in ast.walk(mod):
    if isinstance(node, (ast.ClassDef, ast.FunctionDef)):
        for dec in node.decorator_list:
            if isinstance(dec, ast.Call):
                dec.keywords = [kw for kw in dec.keywords if kw.arg != "pip_install"]

with tempfile.NamedTemporaryFile("w", delete=False, suffix=".py") as temp_file:
    temp_file.write(ast.unparse(mod))
    temp_file_path = temp_file.name

!single-spec run {temp_file_path}
os.unlink(temp_file_path)

Using device: cuda:0
The grounding tool detected two bounding boxes for the target entity "cat" in the image. Here are the coordinates:

1. **First Box**: Top-left (9, 54) to Bottom-right (316, 474)
2. **Second Box**: Top-left (346, 23) to Bottom-right (639, 372)

These boxes represent the detected positions of cats in the image with a confidence threshold of 0.3.


## Run the agent locally while executing the tool inside a Docker container.

In [ ]:
!single-spec run {example_path}

In [27]:
## Clear Cache
!single-spec clear